In [121]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

# Load the dataset
df = pd.read_csv('GHI Final Dataset.csv', encoding='ISO-8859-1')
print(df.head())

       Country  Year Month  Hunger_Index Food_Insecurity_Level  \
0  Afghanistan  2000   Jan          49.6             High Risk   
1  Afghanistan  2000   Feb          49.6             High Risk   
2  Afghanistan  2000   Mar          49.6             High Risk   
3  Afghanistan  2000   Apr          49.6             High Risk   
4  Afghanistan  2000   May          49.6             High Risk   

   Population_Growth  Annual_Precipitation_mm  IDP_Count  
0           1.212176                    327.0        NaN  
1           1.212176                    327.0        NaN  
2           1.212176                    327.0        NaN  
3           1.212176                    327.0        NaN  
4           1.212176                    327.0        NaN  


In [122]:
# Data preprocessing
df = df.drop(columns=["Country", "Month", "Year"])  # Dropping non-relevant columns
df = df.fillna(0)  # Filling missing values with zero

In [123]:
# Identify the categorical column 'Food_Insecurity_Level' and encode it
categorical_columns = ['Food_Insecurity_Level']  # Modify according to your dataset
encoder = LabelEncoder()

In [124]:
# Apply label encoding to the 'Food_Insecurity_Level' column
for col in categorical_columns:
    if col in df.columns:
        df[col] = encoder.fit_transform(df[col])


In [125]:
# Create the target variable 'GHI_Label' based on 'Hunger_Index'
df['GHI_Label'] = (df['Hunger_Index'] // 10).astype(int)
df['GHI_Label'] = df['GHI_Label'].clip(1, 10)  # Ensure values are between 1 and 10
print(df.head())

   Hunger_Index  Food_Insecurity_Level  Population_Growth  \
0          49.6                      0           1.212176   
1          49.6                      0           1.212176   
2          49.6                      0           1.212176   
3          49.6                      0           1.212176   
4          49.6                      0           1.212176   

   Annual_Precipitation_mm  IDP_Count  GHI_Label  
0                    327.0        0.0          4  
1                    327.0        0.0          4  
2                    327.0        0.0          4  
3                    327.0        0.0          4  
4                    327.0        0.0          4  


In [126]:

# Features (X) and target (y)
X = df.drop(columns=['Hunger_Index', 'GHI_Label']).values
y = df['GHI_Label'].values

In [127]:
# Standardize the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [128]:

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [129]:
# Convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


In [130]:
# Define a custom dataset class
class GHI_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [131]:
# Create DataLoader for train and test sets
train_dataset = GHI_Dataset(X_train_tensor, y_train_tensor)
test_dataset = GHI_Dataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [132]:

# Define the transformer model
class TransformerModel(nn.Module):
    def __init__(self, input_dim, output_dim, num_heads=4, hidden_dim=256, num_layers=3):
        super(TransformerModel, self).__init__()
        
        self.embedding = nn.Linear(input_dim, hidden_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_dim, 
                nhead=num_heads, 
                dim_feedforward=hidden_dim
            ), 
            num_layers=num_layers
        )
        self.fc_out = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)  # Add a sequence dimension
        x = self.transformer(x)
        x = x.squeeze(1)  # Remove the sequence dimension
        out = self.fc_out(x)
        return out

In [133]:
# Model parameters
input_dim = X_train.shape[1]
output_dim = 10  # GHI_Label values range from 1 to 10

In [134]:
# Initialize the model
model = TransformerModel(input_dim, output_dim)

In [135]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [136]:

# Training function
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels - 1)  # Subtract 1 since GHI_Label is 1-indexed
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(train_loader)}")


In [137]:
# Evaluate function
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels - 1).sum().item()  # Adjust labels to 0-indexed
            total += labels.size(0)
    
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy * 100:.2f}%")


In [138]:
# Train and evaluate the model
train_model(model, train_loader, criterion, optimizer, num_epochs=50)
evaluate_model(model, test_loader)

Epoch 1/50, Loss: 0.5367180094832465
Epoch 2/50, Loss: 0.50794866564986
Epoch 3/50, Loss: 0.5370345351757941
Epoch 4/50, Loss: 0.5748452435195188
Epoch 5/50, Loss: 0.6164691536586522
Epoch 6/50, Loss: 0.5333569097570526
Epoch 7/50, Loss: 0.5314366049838789
Epoch 8/50, Loss: 0.5310342975638129
Epoch 9/50, Loss: 0.5297847592727446
Epoch 10/50, Loss: 0.549966360647957
Epoch 11/50, Loss: 0.5907306541483124
Epoch 12/50, Loss: 0.5872482707103094
Epoch 13/50, Loss: 0.5680226658458833
Epoch 14/50, Loss: 0.5667076713982082
Epoch 15/50, Loss: 0.5902968568332267
Epoch 16/50, Loss: 0.9483043206331534
Epoch 17/50, Loss: 1.033978636130626
Epoch 18/50, Loss: 0.6634002279538613
Epoch 19/50, Loss: 0.6648253971364075
Epoch 20/50, Loss: 0.6427681478329035
Epoch 21/50, Loss: 0.7697383444675635
Epoch 22/50, Loss: 1.0664420860670345
Epoch 23/50, Loss: 1.1741807427241173
Epoch 24/50, Loss: 1.1668884371008192
Epoch 25/50, Loss: 1.1572461579785203
Epoch 26/50, Loss: 1.1888083312418554
Epoch 27/50, Loss: 1.1994